# 用 PROC LOAN 比较学生贷款还款方案


## 执行摘要

某学生资助办公室要评估一个即将毕业的群体应如何偿还代表性的 $27,500 余额，比较四种还款结构——联邦固定利率标准计划、含手续费的私人再融资计划、有上限的浮动利率（ARM）贷款，以及雇主赞助的利率买断计划——使用 `PROC LOAN`。在120个月的期限内，四种方案按各自报价的起始利率对比：联邦标准计划的利息最高（利率6.53%时为 **$10,021.22**，月供 **$312.68**），私人再融资降低了利率但增加了 $412.50 的手续费（利率5.99%时为 **$9,120.20**，月供 **$305.17**），而报价更低的 ARM（利率4.75%时为 **$7,099.76**，月供 **$288.33**）和雇主买断计划（利率4.50%时为 **$6,700.67**，月供 **$285.01**）的利息总额最小。随后 `COMPARE` 语句块报告了每个方案在第36、60、120个月时的累计利息、累计本金和未偿余额，展示了借款人最可能进行再融资或还清贷款的时点上，每笔贷款的摊销进度。


## 数据来源

| 数据集 | 行数 | 描述 | 关键变量 |
|---------|------|-------------|---------------|
| `borrowers` | 40 | 使用 `call streaminit(20260531)` 和 `rand('uniform')` 内联生成的合成毕业群体贷款档案，用于为比较分析提供切合实际的贷款条件。 | `student_id`（1001-1040）、`balance`（毕业时的本金；本次抽样范围为 $11,800-$47,750，均值 $30,771）、`apr`（4.10%-9.10% 的年名义利率，均值 6.50%）、`term`（120 或 180 个月，均值 144）、`origination_pct`（1.0%-2.0% 的手续费，均值 1.50%） |

用 `PROC LOAN` 分析的代表性借款人（金额 $27,500，期限120个月，2026年7月起）在该群体的余额和利率分布中处于中下位置；未使用任何外部或网络数据。该群体的作用是为贷款条件提供合理依据——正面比较则是在这一笔代表性贷款上进行的。


# 用 PROC LOAN 比较学生贷款还款方案

学生毕业时，学生资助办公室必须帮助他们在各种竞争性还款方案中做出选择。`PROC LOAN`（SAS/ETS）正是为此而设计：它能够摊销固定利率、浮动利率（ARM）和买断型贷款，并在月供、总利息和摊销进度上对它们进行并排比较。

在本笔记中，我们将：

1. 生成一个合成毕业群体，以确定切合实际的贷款条件。
2. 使用 `PROC MEANS` 汇总该群体。
3. 为一笔代表性的 $27,500 余额构建四种还款方案，并用 `PROC LOAN`（FIXED / ARM / BUYDOWN + COMPARE）进行比较。
4. 单独重新运行推荐的固定利率方案，以确认其独立的经济性。

全部计算均在本地基于内联合成数据完成——不使用任何外部文件或网络访问。


## 步骤 1 —— 生成合成毕业群体

我们模拟 40 名借款人。每人在毕业时抽取一个本金余额、一个与信用等级大致相关的年利率、一个标准还款期限（10 年或 15 年），以及一个手续费百分比。种子值使数据可重复。


In [1]:
数据 borrowers;
   调用 streaminit(20260531);
   长度 plan $ 28;
   循环 student_id = 1001 到 1040;
      /* Principal balance at graduation: $9,500 - $48,000 */
      balance = round(9500 + 38500*rand('uniform'), 50);
      /* Annual nominal APR by credit tier: 4.0% - 9.5% */
      apr = round(4.0 + 5.5*rand('uniform'), 0.05);
      /* Standard repayment term: 120 or 180 months */
      如果 rand('uniform') < 0.6 那么 term = 120;
      否则 term = 180;
      /* Origination fee as a percent of principal: 1.0% - 2.0% */
      origination_pct = round(1.0 + rand('uniform'), 0.10);
      输出;
   结束;
运行;


NOTE: DATA borrowers


NOTE: Wrote borrowers (40 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## 步骤 2 —— 分析该群体的概况

在对个别贷款建模之前，我们先查看余额、利率和期限的分布情况。这能告诉我们*代表性*借款人的样貌——也是后续正面比较的基础。


In [2]:
过程 均值 数据=borrowers n mean MIN MAX maxdec=2;
   变量 balance apr term origination_pct;
运行;

                                                  The MEANS Procedure

 Variable               N           Mean     Minimum     Maximum
 ---------------------------------------------------------------
 balance               40       30771.25    11800.00    47750.00
 apr                   40           6.50        4.10        9.10
 term                  40         144.00      120.00      180.00
 origination_pct       40           1.50        1.00        2.00
 ---------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 步骤 3 —— 为代表性借款人比较四种还款方案

以一笔代表性的 $27,500 余额、120 个月（10 年）期限、2026 年 7 月开始计算，我们列出四种切合实际的方案：

- **联邦直接无补贴贷款（标准）** —— 利率为 6.53% 的普通固定利率贷款。
- **私人再融资（含手续费）** —— 更低的 5.99% 固定利率，但需支付 $412.50 的初始化费用（`INIT=`）。
- **私人浮动利率贷款（ARM）** —— 利率为 4.75% 的可调整贷款，带有每次调整 1%／终身 5% 的 `CAPS=` 上限、9.75% 的 `MAXRATE=`、每年一次的 `ADJUSTFREQ=`，以及 `WORSTCASE` 关键字。
- **雇主 2-1 买断计划** —— 报价为补贴后 4.50% 的起始利率，按报价时间表通过 `BDRATES=` 在第 2 年升至 5.50%，此后为 6.50%。

`COMPARE` 语句要求在第 36、60、120 个月给出跨贷款视图，`TAXRATE=` 为 22%，最低有吸引力回报率（`MARR=`）为 4%；`OUTSUM=` 和 `OUTCOMP=` 分别捕获每笔贷款的汇总和比较行。下面的清单报告了每个方案在每个检查点的**累计已付利息、累计已还本金和未偿余额**——清晰展示了各候选方案的摊销进度。

> **关于利率调整的说明。** Jenner 的 `PROC LOAN` 按每个方案报价的**起始**利率摊销，因此 ARM 的 `CAPS`／`MAXRATE`／`WORSTCASE` 设置以及买断计划的 `BDRATES` 阶梯利率虽会在清单中作为贷款条款回显，但**不会**被应用到还款计算中——下面的 ARM 和买断计划数字反映的是其 4.75% 和 4.50% 的起始利率在整个期限内保持不变。请将这两项总额视为最佳情形（起始利率）成本，而非最差情形上限。


In [3]:
过程 loan START=2026:7 amount=27500 life=120 outsum=plan_sum;

   fixed rate=6.53
         label='联邦直接无补贴贷款（标准）';

   fixed rate=5.99 init=412.50
         label='私人再融资（含手续费）';

   arm rate=4.75 caps=(1 5) maxrate=9.75 adjustfreq=12 worstcase
       label='私人浮动利率贷款（最差情形）';

   buydown rate=4.50 bdrates=(13=5.50 25=6.50)
           label='雇主2-1买断计划后6.5%';

   COMPARE at=(36 60 120) ALL taxrate=22 marr=4 OUTCOMP=plan_cmp;
运行;

                            The LOAN Procedure

  Number of loans evaluated:    4

  Loan #1: 联邦直接无补贴贷款（标准）
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan #2: 私人再融资（含手续费）
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            5.9900
    Life (months):                          120
    Monthly Payment:                     305.17
    Total Paid (P&I):                  36620.20
    Total Interest:                     9120.20
    Initialization Cost:                 412.50

  Loan #3: 私人浮动利率贷款（最差情形）
    Loan Type:                   Adjustable Rate
    Amount (Principal):                27500.00
    Interest Rate (annual %):            4


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## 步骤 4 —— 单独重新运行推荐的固定利率方案

对于看重还款确定性的借款人，联邦固定利率标准计划是安全的默认选择。我们单独重新运行该方案，以确认其核心经济指标：与四方案比较中相同的 **$312.68** 月供、**$37,521.22** 总还款额和 **$10,021.22** 总利息，现在以独立贷款汇总的形式呈现。


In [4]:
过程 loan START=2026:7 amount=27500 rate=6.53 life=120 schedule=yearly;
   fixed label='联邦直接无补贴贷款（标准）';
运行;

                            The LOAN Procedure

  Number of loans evaluated:    1

  Loan #1: 联邦直接无补贴贷款（标准）
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan Repayment Schedule: 联邦直接无补贴贷款（标准）
      Year    Begin Balance        Payment       Interest      Principal      End Balance
    ------ ---------------- -------------- -------------- -------------- ----------------
         1         27500.00        3752.12        1736.12        2016.00         25484.00
         2         25484.00        3752.12        1600.47        2151.66         23332.34
         3         23332.34        3752.12        1455.68        2296.44         21035.90
         4         21035.90        3752.12        1301.15        2450.97 


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## 结果解读

四个方案都在 120 个月内完全摊销了相同的 $27,500 本金（每个方案的未偿余额在第 120 个月时都降至 $0.00，见 COMPARE 表），因此决策的关键在于**月供**和**报价利率下的总利息**：

| 方案 | 利率 | 月供 | 总利息 | 备注 |
|------|------|---------|----------------|-------|
| 联邦直接标准贷款 | 6.53% | $312.68 | $10,021.22 | 无手续费；借款人保护最强 |
| 私人再融资 | 5.99% | $305.17 | $9,120.20 | $412.50 的手续费 |
| 私人浮动利率贷款（ARM） | 4.75%（起始） | $288.33 | $7,099.76 | 利率可能上升；此数字为起始利率成本 |
| 雇主 2-1 买断计划 | 4.50%（起始） | $285.01 | $6,700.67 | 取决于是否持续在职 |

- **联邦标准**计划在利息上最贵（$10,021.22），但提供固定、可预测的 $312.68 月供，且无手续费。
- **私人再融资**将利率降至 5.99%（利息 $9,120.20，比联邦计划少 $901），但收取 $412.50 的初始手续费，因此相对联邦计划的净优势大约是利息差减去手续费后约 $488——只有当贷款持续足够长时间、不被提前再融资掉时才有意义。
- **ARM** 和**买断计划**在这里显示的利息最低（分别为 $7,099.76 和 $6,700.67），纯粹是因为它们的**起始**利率远低于固定利率方案。正如步骤 3 中所述，Jenner 会将这些起始利率保持不变，因此这些是最佳情形数字：实际会上调的 ARM，或失去雇主补贴的买断计划，其成本会更高，且月供并非有保障。

`COMPARE` 表通过展示每个方案的摊销速度进一步说明了这一点。在第 36 个月，联邦计划已支付 $4,792.27 的利息，偿还了 $6,464.10 的本金（余额 $21,035.90），而买断计划只支付了 $3,263.97 的利息，但偿还了 $6,996.24 的本金（余额 $20,503.76）——利率更低的方案在头三年内既支付更少利息，本金削减速度也更快。对于风险规避型毕业生，联邦标准计划的利率确定性往往能证明其更高利息是合理的；而对于确信自己会长期稳定就业的借款人，买断计划的补贴起始利率能在真正锁定利率的选项中带来最大的节省。
